<h1>CatBoost with Scada Dataset</h1>

Currently the CatBoost Forecaster can only be trained with one timeseries (here: turbine) at a time. It should be possible to train it on multiple turbines at once, this will probably be implemented in the future.

In [1]:
# Set up Spark

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SCADA-Forecasting")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

# Load Data

import pandas as pd
from sktime.forecasting.model_selection import temporal_train_test_split

pdf = pd.read_parquet(r"scada_prepro.parquet")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/16 16:18:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [12]:
pdf_turbine1 = pdf[pdf["item_id"] == "1_Kelmarsh"].copy()
pdf_turbine1 = pdf_turbine1.drop(columns=["item_id"])
pdf_turbine1["timestamp"] = pd.to_datetime(pdf_turbine1["timestamp"])
pdf_turbine1 = pdf_turbine1.set_index("timestamp").sort_index()
pdft = pdf_turbine1[3:]
pdf_turbine1_no_NaN = pdf_turbine1.dropna(axis=1)
pdf_turbine1_no_NaN = pdf_turbine1_no_NaN.asfreq("10min")
pdf_turbine1_no_NaN["target"] = pdf_turbine1_no_NaN["target"].interpolate("time").ffill().bfill()
X_cols = [c for c in pdf_turbine1_no_NaN.columns if c != "target"]
pdf_turbine1_no_NaN[X_cols] = pdf_turbine1_no_NaN[X_cols].interpolate("time").ffill().bfill()


train_pdf, test_pdf = temporal_train_test_split(pdf_turbine1_no_NaN, test_size=0.2)
y_train = train_pdf["target"]

y_test = test_pdf["target"]
X_test= test_pdf.drop(columns=["target"])

In [13]:
len(pdf_turbine1)

52236

In [14]:
import sys
sys.path.append('/workspaces/amos2025ws03-rtdip-timeseries-forecasting/src/sdk/python')
from rtdip_sdk.pipelines.forecasting.spark.catboost_timeseries import CatboostTimeSeries

In [15]:
cbts = CatboostTimeSeries(target_col="target", timestamp_col="timestamp")

In [16]:
cbts.train(spark.createDataFrame(train_pdf.reset_index()))

0:	learn: 648.1804997	total: 842ms	remaining: 3m 29s
1:	learn: 619.1597785	total: 1.52s	remaining: 3m 8s
2:	learn: 591.5538326	total: 2.22s	remaining: 3m 3s
3:	learn: 565.3112375	total: 2.92s	remaining: 2m 59s
4:	learn: 540.3151533	total: 3.67s	remaining: 3m
5:	learn: 516.3928463	total: 4.41s	remaining: 2m 59s
6:	learn: 494.2503217	total: 5.17s	remaining: 2m 59s
7:	learn: 472.9118067	total: 5.92s	remaining: 2m 59s
8:	learn: 452.5242805	total: 6.61s	remaining: 2m 56s
9:	learn: 433.5954722	total: 7.27s	remaining: 2m 54s
10:	learn: 415.5698927	total: 7.95s	remaining: 2m 52s
11:	learn: 398.5450693	total: 8.62s	remaining: 2m 50s
12:	learn: 382.6125773	total: 9.3s	remaining: 2m 49s
13:	learn: 367.4884276	total: 10s	remaining: 2m 48s
14:	learn: 352.8734485	total: 10.7s	remaining: 2m 47s
15:	learn: 339.3263482	total: 11.3s	remaining: 2m 45s
16:	learn: 326.3954555	total: 12s	remaining: 2m 44s
17:	learn: 314.1838754	total: 12.7s	remaining: 2m 43s
18:	learn: 302.7153239	total: 13.4s	remaining: 2m

In [17]:
spark_test = spark.createDataFrame(test_pdf.reset_index())
metrics = cbts.evaluate(spark_test)

Evaluated on 10541 predictions

Catboost Metrics:
--------------------------------------------------------------------------------
MAE                 : 104.7876
RMSE                : 154.5119
MAPE                : 5.2792
MASE                : 1.0964
SMAPE               : 111.9547

MAE_r               : 118.5180
RMSE_r              : 169.4848
MAPE_r              : 14.6246
MASE_r              : 1.2231
SMAPE_r             : 115.1608


In [3]:
import sys
sys.path.append('/workspaces/amos2025ws03-rtdip-timeseries-forecasting/src/sdk/python')
from rtdip_sdk.pipelines.forecasting.spark.catboost_timeseries_refactored import CatBoostTimeSeries
from rtdip_sdk.pipelines.forecasting.spark.xgboost_timeseries import XGBoostTimeSeries
from rtdip_sdk.pipelines.forecasting.spark.lstm_timeseries import LSTMTimeSeries
from rtdip_sdk.pipelines.forecasting.spark.autogluon_timeseries import AutoGluonTimeSeries

In [12]:
df = pd.read_parquet(r"scada_prepro.parquet")

df = df.sort_values(["item_id", "timestamp"])

train_dfs = []
test_dfs = []
for item_id in df["item_id"].unique():
    item_data = df[df["item_id"] == item_id]
    split_idx = int(len(item_data) * 0.8)
    train_dfs.append(item_data.iloc[:split_idx])
    test_dfs.append(item_data.iloc[split_idx:])

train_df = pd.concat(train_dfs, ignore_index=True)
test_df = pd.concat(test_dfs, ignore_index=True)

spark = SparkSession.builder.getOrCreate()
train_spark = spark.createDataFrame(train_df)
test_spark = spark.createDataFrame(test_df)

<h2>XGBoost x Scada </h2>

In [17]:
xgb = XGBoostTimeSeries()
xgb.train(train_spark)
xgb.evaluate(test_spark)

TRAINING XGBOOST MODEL


26/01/16 16:29:56 WARN TaskSetManager: Stage 3 contains a task of very large size (7774 KiB). The maximum recommended task size is 1000 KiB.


Training data: 250,750 rows, 6 sensors
Engineering features
After removing NaN rows: 250,462 rows

Training XGBoost with 250,462 samples
Features: ['sensor_encoded', 'hour', 'day_of_week', 'day_of_month', 'month', 'lag_1', 'lag_6', 'lag_12', 'lag_24', 'lag_48', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24']
Model parameters:
  max_depth: 6
  learning_rate: 0.1
  n_estimators: 100
  n_jobs: -1

Training completed

Top 5 Most Important Features:
        feature  importance
          lag_1    0.962846
rolling_mean_12    0.015360
 rolling_std_12    0.004983
          lag_6    0.002299
         lag_12    0.002055
EVALUATING XGBOOST MODEL


26/01/16 16:30:00 WARN TaskSetManager: Stage 4 contains a task of very large size (4273 KiB). The maximum recommended task size is 1000 KiB.


Engineering features
Test samples: 62,402
Evaluated on 62402 predictions

XGBoost Metrics:
--------------------------------------------------------------------------------
MAE                 : 93.1255
RMSE                : 144.3044
MAPE                : 0.9579
MASE                : 0.9930
SMAPE               : 44.4073

MAE_r               : 94.7372
RMSE_r              : 141.7921
MAPE_r              : 0.8099
MASE_r              : 0.9826
SMAPE_r             : 38.7626


{'MAE': -93.12549253289968,
 'RMSE': -144.30440743903947,
 'MAPE': -0.9579473336614199,
 'MASE': -0.9930167757483279,
 'SMAPE': -44.40732572343409}

<h2>CatBoost x Scada</h2>

In [18]:
cb = CatBoostTimeSeries()
cb.train(train_spark)
cb.evaluate(test_spark)

TRAINING XGBOOST MODEL


26/01/16 16:30:00 WARN TaskSetManager: Stage 5 contains a task of very large size (7774 KiB). The maximum recommended task size is 1000 KiB.


Training data: 250,750 rows, 6 sensors
Engineering features
After removing NaN rows: 250,462 rows

Training XGBoost with 250,462 samples
Features: ['sensor_encoded', 'hour', 'day_of_week', 'day_of_month', 'month', 'lag_1', 'lag_6', 'lag_12', 'lag_24', 'lag_48', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24']
Model parameters:
  max_depth: 6
  learning_rate: 0.1
  n_estimators: 100
  n_jobs: -1

Training completed

Top 5 Most Important Features:
        feature  importance
          lag_1   79.580074
rolling_mean_12   10.114654
 rolling_std_12    4.768764
rolling_mean_24    1.424556
         lag_12    1.391971
EVALUATING XGBOOST MODEL


26/01/16 16:30:04 WARN TaskSetManager: Stage 6 contains a task of very large size (4273 KiB). The maximum recommended task size is 1000 KiB.


Engineering features
Test samples: 62,402
Evaluated on 62402 predictions

XGBoost Metrics:
--------------------------------------------------------------------------------
MAE                 : 93.7749
RMSE                : 145.2295
MAPE                : 1.1548
MASE                : 0.9999
SMAPE               : 45.7723

MAE_r               : 95.4867
RMSE_r              : 142.7610
MAPE_r              : 0.8686
MASE_r              : 0.9904
SMAPE_r             : 39.9239


{'MAE': -93.77493113168842,
 'RMSE': -145.2295294305284,
 'MAPE': -1.1547528996683498,
 'MASE': -0.9999418765545102,
 'SMAPE': -45.772286275208}

In [20]:
lstm = LSTMTimeSeries(epochs=5)
lstm.train(train_spark)
lstm.evaluate(test_spark)

TRAINING LSTM MODEL (SINGLE MODEL WITH EMBEDDINGS)


26/01/16 16:42:32 WARN TaskSetManager: Stage 8 contains a task of very large size (7774 KiB). The maximum recommended task size is 1000 KiB.


Training single model for 6 sensors
Total training samples: 250750
Configuration: 2 LSTM layers, 64 units each
Sensor embedding dimension: 8
Lookback window: 168, Forecast horizon: 24

Creating training sequences
Created 249604 training sequences
Input shape: (249604, 168, 1), Sensor IDs shape: (249604, 1), Output shape: (249604, 24)

Building model


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sensor_input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sensor_embedding    │ (None, 1, 8)      │         48 │ sensor_input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 8)         │          0 │ sensor_embedding… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ values_input        │ (None, 168, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_1     │ (None, 168, 8)    │          0 │ flatten_1[0][0]   │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 168, 9)    │          0 │ values_input[0][… │
│ (Concatenate)       │                   │            │ repeat_vector_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 168, 64)   │     18,944 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 168, 64)   │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 64)        │     33,024 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 24)        │      1,560 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 53,576 (209.28 KB)

 Trainable params: 53,576 (209.28 KB)

 Non-trainable params: 0 (0.00 B)

None

Training model
Epoch 1/5
 146/6241 ━━━━━━━━━━━━━━━━━━━━ 7:42 76ms/step - loss: 0.5335 - mae: 0.5640

KeyboardInterrupt: 